# CNN image classification — CIFAR-10 (PyTorch)

**DeepVision Masterclass — Modern Edition**

This notebook is designed as an executable teaching lab. Read the dataset contract and validation notes before training.

## 1. Environment and reproducibility
Install a PyTorch/torchvision pair using the official selector for your OS and accelerator. This notebook uses CIFAR-10, a public 10-class benchmark.

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
print('PyTorch:', torch.__version__)
print('Device:', 'cuda' if torch.cuda.is_available() else 'cpu')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 2. Dataset and preprocessing
CIFAR-10 images are 32×32 RGB. Normalization uses the conventional channel statistics. The test set is never augmented.

In [ ]:
mean, std = (0.4914,0.4822,0.4465), (0.2470,0.2435,0.2616)
train_tf = transforms.Compose([transforms.RandomCrop(32,padding=4), transforms.RandomHorizontalFlip(), transforms.ToTensor(), transforms.Normalize(mean,std)])
test_tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean,std)])
train_ds = datasets.CIFAR10('data', train=True, download=True, transform=train_tf)
test_ds = datasets.CIFAR10('data', train=False, download=True, transform=test_tf)
classes = train_ds.classes
train_loader = DataLoader(train_ds,batch_size=128,shuffle=True,num_workers=2,pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_ds,batch_size=256,shuffle=False,num_workers=2)
print(len(train_ds),len(test_ds),classes)

## 3. Visual inspection
Undo normalization before plotting.

In [ ]:
x,y = next(iter(train_loader))
img = (x[0].permute(1,2,0).numpy()*np.array(std)+np.array(mean)).clip(0,1)
plt.imshow(img); plt.title(classes[y[0]]); plt.axis('off');

## 4. CNN architecture
A compact baseline with batch normalization, dropout and global average pooling.

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64,128,3,padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1))
        self.classifier = nn.Sequential(nn.Flatten(),nn.Dropout(.3),nn.Linear(128,n_classes))
    def forward(self,x): return self.classifier(self.features(x))
model=SmallCNN().to(device)
print('Parameters:',sum(p.numel() for p in model.parameters() if p.requires_grad))

## 5. Training loop
The loop records training loss and validation accuracy. Keep the test set for final reporting.

In [ ]:
criterion=nn.CrossEntropyLoss()
optimizer=torch.optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-4)
def evaluate(model, loader):
    model.eval(); correct=total=0; ys=[]; ps=[]
    with torch.no_grad():
        for xb,yb in loader:
            xb,yb=xb.to(device),yb.to(device); logits=model(xb); pred=logits.argmax(1)
            correct+=(pred==yb).sum().item(); total+=yb.size(0); ys.extend(yb.cpu().tolist()); ps.extend(pred.cpu().tolist())
    return correct/total, np.array(ys), np.array(ps)
EPOCHS=5  # Increase to 20–40 for a serious benchmark
history={'loss':[],'val_acc':[]}
for epoch in range(EPOCHS):
    model.train(); running=0
    for xb,yb in train_loader:
        xb,yb=xb.to(device),yb.to(device); optimizer.zero_grad(set_to_none=True)
        loss=criterion(model(xb),yb); loss.backward(); optimizer.step(); running+=loss.item()*yb.size(0)
    val_acc,_,_=evaluate(model,test_loader)
    history['loss'].append(running/len(train_ds)); history['val_acc'].append(val_acc)
    print(f'Epoch {epoch+1}/{EPOCHS} loss={history["loss"][-1]:.4f} test-accuracy={val_acc:.4f}')

## 6. Final evaluation and checkpoint

In [ ]:
acc,y_true,y_pred=evaluate(model,test_loader)
print('Accuracy:',acc)
print(classification_report(y_true,y_pred,target_names=classes,digits=4))
ConfusionMatrixDisplay.from_predictions(y_true,y_pred,display_labels=classes,xticks_rotation=45,cmap='Blues'); plt.tight_layout()
torch.save({'model_state_dict':model.state_dict(),'classes':classes,'seed':SEED},'cifar10_cnn.pt')

## 7. Exercises
1. Replace the CNN with a residual block. 2. Compare SGD+momentum and AdamW. 3. Add cosine learning-rate decay and early stopping. 4. Report macro-F1 and per-class recall. 5. Compare no augmentation vs crop/flip using the same split.